# LiTFiC → How2Sign (Vid+Prev) demo — Kaggle T4
**Before running:** right panel → Settings → Accelerator → **GPU T4 x2** (or x1); Internet **On**. Do **not** install flash-attn. Run cells top-to-bottom. All `!` commands are single-line on purpose (multi-line `\` continuation breaks inside Kaggle `!` cells).

In [ ]:
# Cell 1 — Clone code
!git clone -b how2sign-t4-demo https://github.com/sonlamhg/LiTFiC.git /kaggle/working/LiTFiC
%cd /kaggle/working/LiTFiC
!git log --oneline -3

In [ ]:
# Cell 2 — Install deps (single line; no flash-attn) + verify
!pip install -q lightning==2.3.0 torchmetrics hydra-core==1.3.2 hydra-colorlog==1.2.0 omegaconf rich rootutils einops lmdb transformers==4.45.2 peft==0.12.0 sentencepiece lightning-utilities==0.11.2 nltk pycocoevalcap lightning-bolts
import lightning, hydra, transformers, peft, lightning_bolts
print("deps OK", lightning.__version__, transformers.__version__)

In [ ]:
# Cell 3 — Env + episode-index files
import os, json
os.environ["PROJECT_ROOT"] = "/kaggle/working/LiTFiC"
for p in ["val_start_indices.json", "test_start_indices.json"]:
    json.dump({"idx": [0]}, open(p, "w"))
OVR = "paths.h2s_tsv_dir=/kaggle/working/LiTFiC/data/tsv paths.h2s_feats_dir=/kaggle/temp/feats"
print("env ready")

In [ ]:
# Cell 4 — Download data (train.zip ~7.78GB, ~15-20 min)
import os
os.makedirs("/kaggle/working/LiTFiC/data/tsv", exist_ok=True)
os.makedirs("/kaggle/temp/feats", exist_ok=True)
B = "https://dataverse.csuc.cat/api/access/datafile"
!wget -q "$B/51923" -O /kaggle/working/LiTFiC/data/tsv/cvpr23.fairseq.i3d.train.how2sign.tab
!wget -q "$B/53222" -O /kaggle/working/LiTFiC/data/tsv/cvpr23.fairseq.i3d.test.how2sign.tab
!wget -q "$B/51543" -O /kaggle/temp/train.zip && unzip -q -o /kaggle/temp/train.zip -d /kaggle/temp/feats/ && rm /kaggle/temp/train.zip
!wget -q "$B/51538" -O /kaggle/temp/test.zip  && unzip -q -o /kaggle/temp/test.zip  -d /kaggle/temp/feats/ && rm /kaggle/temp/test.zip
!echo "n_npy=$(find /kaggle/temp/feats -name '*.npy' | wc -l)"

In [ ]:
# Cell 5 — Verify schema (READ OUTPUT: dim should be 1024, one .npy per sentence)
import csv, glob, os, numpy as np
tab = "/kaggle/working/LiTFiC/data/tsv/cvpr23.fairseq.i3d.train.how2sign.tab"
r = csv.DictReader(open(tab, newline="", encoding="utf-8"), delimiter="	")
print("COLUMNS:", r.fieldnames)
row = next(r); print("id=", row.get("id"), "| translation=", row.get("translation"))
print("signs_file=", row.get("signs_file"), "| offset/len=", row.get("signs_offset"), row.get("signs_length"))
npy = glob.glob("/kaggle/temp/feats/**/*.npy", recursive=True)[0]
print("NPY", os.path.basename(npy), np.load(npy).shape)

In [ ]:
# Cell 6 — M1 sanity (no OOM, finite loss). OOM? append model.net.mm_projector_config.projector_type=conv_K5_P2_KP2_L2
!python src/train.py experiment=how2sign-vid {OVR} trainer.max_epochs=1 +trainer.limit_train_batches=20 +trainer.limit_val_batches=5 logger=csv

In [ ]:
# Cell 7 — M1 train (Vid-only, 3 epochs)
!python src/train.py experiment=how2sign-vid {OVR} trainer.max_epochs=3 callbacks.model_checkpoint.every_n_train_steps=500 logger=csv

In [ ]:
# Cell 8 — M1 eval on test subset
import glob, os
CKPT = sorted(glob.glob("/kaggle/working/LiTFiC/**/*.ckpt", recursive=True), key=os.path.getmtime)[-1]
print("using", CKPT)
!python src/eval.py experiment=how2sign-vid {OVR} ckpt_path="{CKPT}" +trainer.limit_test_batches=50 logger=csv

In [ ]:
# Cell 9 — M2 train (Vid+Prev, 3 epochs)
!python src/train.py experiment=how2sign-vid+prev {OVR} trainer.max_epochs=3 callbacks.model_checkpoint.every_n_train_steps=500 logger=csv

In [ ]:
# Cell 10 — M2 eval + ablation (compare BLEU-4/ROUGE-L vs M1 on same subset)
import glob, os
CKPT2 = sorted(glob.glob("/kaggle/working/LiTFiC/**/*.ckpt", recursive=True), key=os.path.getmtime)[-1]
print("using", CKPT2)
!python src/eval.py experiment=how2sign-vid+prev {OVR} ckpt_path="{CKPT2}" +trainer.limit_test_batches=50 logger=csv